# 🔄 Preprocess Only - Batch Processing

**Process crawled VifactCheck data with memory-efficient batch processing**

📥 Input: `./src/data/json/news_data_vifactcheck_*.json`
📤 Output: `./processed_data/crawled/`


In [4]:
# Install dependencies
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "torch",
    "torchvision",
    "transformers",
    "pillow",
    "matplotlib",
    "tqdm",
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"📦 Installed {pkg}")

✅ pandas
✅ numpy
✅ torch
✅ torchvision


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers
📦 Installed pillow
✅ matplotlib
✅ tqdm


In [5]:
# Setup
import sys
import os
import orjson
import numpy as np
import torch
import gc
from pathlib import Path
from tqdm import tqdm

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"✅ Ready: {project_root}")

✅ Ready: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection


In [8]:
# Configuration
PREPROCESSING_CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet50",
    "language": "vi",
    "max_length": 512,
    "batch_size": 32,  # Batch size for processing
    "save_format": "npz",
    "input_dir": "./data/json",
    "output_dir": "./processed_data/crawled",
}

os.makedirs(PREPROCESSING_CONFIG["output_dir"], exist_ok=True)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {device}")
print(f"📦 Batch size: {PREPROCESSING_CONFIG['batch_size']}")

🔧 Device: mps
📦 Batch size: 32


In [7]:
# Initialize preprocessor
try:
    from src.preprocessing import CombinedPreprocessor

    preprocessor = CombinedPreprocessor(
        text_model_name=PREPROCESSING_CONFIG["text_model"],
        image_model_name=PREPROCESSING_CONFIG["image_model"],
        language=PREPROCESSING_CONFIG["language"],
        max_text_length=PREPROCESSING_CONFIG["max_length"],
        device=device,
    )
    print("✅ Preprocessor ready")
except Exception as e:
    print(f"❌ Preprocessor error: {e}")

/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/haila/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 86.3MB/s]


✅ Preprocessor ready


In [5]:
# Load crawled data
def load_data():

    data_dir = Path(PREPROCESSING_CONFIG["input_dir"])
    if not data_dir.exists():
        print(f"❌ Input directory not found: {data_dir}")
        return []

    articles = []
    json_files = list(data_dir.glob("news_data_vifactcheck_*.json"))

    for json_file in json_files:
        with open(json_file, "rb") as f:  # orjson requires binary mode
            content = f.read()
            data = orjson.loads(content)  # use loads(), not load()
            articles.extend(data)
            print(f"✅ Loaded {len(data)} from {json_file.name}")

    return articles


crawled_articles = load_data()
print(f"\n📊 Total articles: {len(crawled_articles)}")

✅ Loaded 1282 from news_data_vifactcheck_test.json
✅ Loaded 634 from news_data_vifactcheck_dev.json
✅ Loaded 4509 from news_data_vifactcheck_train.json

📊 Total articles: 6425


In [6]:
# Prepare data
if crawled_articles:
    texts = []
    image_paths = []
    labels = []

    for article in crawled_articles:
        text = (
            article.get("text", "")
            or article.get("content", "")
            or article.get("title", "")
        )
        if text.strip():
            texts.append(text.strip())
            images = article.get("images", [])

            # Extract image path
            if images and len(images) > 0:
                if isinstance(images[0], str):
                    img_path = images[0]
                elif isinstance(images[0], dict):
                    img_path = images[0].get("url", "") or images[0].get("path", "")
                else:
                    img_path = str(images[0])
                image_paths.append(img_path if img_path else None)
            else:
                image_paths.append(None)

            # Extract label
            label = article.get("label", 0)
            if isinstance(label, str):
                label = 1 if label.lower() in ["fake", "false", "1"] else 0
            labels.append(int(label))

    print(f"📊 Prepared {len(texts)} samples")
    print(f"🏷️ Labels: {len(set(labels))} classes")
else:
    print("❌ No data available")

📊 Prepared 4814 samples
🏷️ Labels: 1 classes


In [7]:
# Create placeholder images
if "texts" in locals() and texts:
    from PIL import Image

    placeholder_dir = Path("./placeholder_images")
    placeholder_dir.mkdir(exist_ok=True)

    processed_paths = []
    for i, img_path in enumerate(image_paths):
        if not img_path or not os.path.exists(img_path):
            placeholder_path = placeholder_dir / f"placeholder_{i}.jpg"
            if not placeholder_path.exists():
                placeholder_array = np.random.randint(
                    128, 200, (224, 224, 3), dtype=np.uint8
                )
                Image.fromarray(placeholder_array).save(placeholder_path)
            processed_paths.append(str(placeholder_path))
        else:
            processed_paths.append(img_path)

    print(f"✅ Created {len(processed_paths)} image paths")

✅ Created 4814 image paths


In [8]:
# Batch preprocessing
if "preprocessor" in locals() and "texts" in locals() and texts:
    BATCH_SIZE = PREPROCESSING_CONFIG["batch_size"]
    total_samples = len(texts)
    num_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"🔄 Starting batch preprocessing...")
    print(f"📊 Total samples: {total_samples}")
    print(f"🔢 Batches: {num_batches}")

    all_text_features = []
    all_image_features = []
    all_labels = []

    for batch_idx in tqdm(range(num_batches), desc="Processing batches"):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, total_samples)

        # Get batch
        batch_texts = texts[start_idx:end_idx]
        batch_image_paths = processed_paths[start_idx:end_idx]
        batch_labels = labels[start_idx:end_idx]

        # Process features
        text_features = preprocessor.text_preprocessor.extract_token_embeddings(
            batch_texts
        )
        image_features = preprocessor.image_preprocessor.extract_features(
            batch_image_paths
        )

        # Collect
        all_text_features.append(text_features)
        all_image_features.append(image_features)
        all_labels.extend(batch_labels)

        # Clear memory
        del text_features, image_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()

    print("🔄 Concatenating features...")
    text_features_all = np.vstack(all_text_features)
    image_features_all = np.vstack(all_image_features)
    labels_all = np.array(all_labels)

    print(
        f"✅ Features shape: Text {text_features_all.shape}, Image {image_features_all.shape}"
    )

🔄 Starting batch preprocessing...
📊 Total samples: 4814
🔢 Batches: 151


Processing batches: 100%|██████████| 151/151 [24:42<00:00,  9.82s/it]


🔄 Concatenating features...
✅ Features shape: Text (4814, 512, 768), Image (4814, 2048)


In [9]:
## 9. Convert NPZ to HDF5 for Training

# Convert the NPZ file to HDF5 format with train/val/test splits for memory-efficient loading during training.
from pathlib import Path
from sklearn.model_selection import train_test_split
import h5py
import numpy as np

NPZ_PATH = Path(PREPROCESSING_CONFIG["output_dir"]) / "combined_dataset.npz"
HDF5_PATH = Path(PREPROCESSING_CONFIG["output_dir"]) / "dataset.h5"


def convert_npz_to_hdf5(npz_path, hdf5_path, train_frac=0.7, val_frac=0.15, seed=42):
    """Convert NPZ to HDF5 with stratified splits."""
    print(f"Loading NPZ: {npz_path}")
    data = np.load(npz_path)

    text_features = data["text_features"]
    image_features = data["image_features"]
    labels = data["labels"]
    n_samples = len(labels)

    print(
        f"Samples: {n_samples}, Text: {text_features.shape}, Image: {image_features.shape}"
    )

    # Stratified split
    indices = np.arange(n_samples)
    train_idx, temp_idx = train_test_split(
        indices, test_size=(1 - train_frac), random_state=seed, stratify=labels
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=seed, stratify=labels[temp_idx]
    )

    print(f"Splits: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    # Create HDF5
    print(f"Creating HDF5: {hdf5_path}")
    with h5py.File(hdf5_path, "w") as f:
        f.create_dataset("train_idx", data=train_idx)
        f.create_dataset("val_idx", data=val_idx)
        f.create_dataset("test_idx", data=test_idx)

        chunk_size = min(64, n_samples)

        f.create_dataset(
            "text_features",
            data=text_features,
            chunks=(chunk_size, text_features.shape[1], text_features.shape[2]),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset(
            "image_features",
            data=image_features,
            chunks=(chunk_size, image_features.shape[1]),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset("labels", data=labels)

        f.attrs["n_samples"] = n_samples
        f.attrs["train_size"] = len(train_idx)
        f.attrs["val_size"] = len(val_idx)
        f.attrs["test_size"] = len(test_idx)

    print("✅ HDF5 created successfully")
    print(f"  NPZ:  {npz_path.stat().st_size / 1e6:.1f} MB")
    print(f"  HDF5: {hdf5_path.stat().st_size / 1e6:.1f} MB")


# Run conversion
if NPZ_PATH.exists():
    convert_npz_to_hdf5(NPZ_PATH, HDF5_PATH)
else:
    print(f"❌ NPZ not found: {NPZ_PATH}")

Loading NPZ: processed_data/crawled/combined_dataset.npz
Samples: 4814, Text: (4814, 512, 768), Image: (4814, 2048)
Splits: train=3369, val=722, test=723
Creating HDF5: processed_data/crawled/dataset.h5
✅ HDF5 created successfully
  NPZ:  7611.2 MB
  HDF5: 5291.4 MB


In [10]:
# Save results
import json

if "text_features_all" in locals():
    save_dir = PREPROCESSING_CONFIG["output_dir"]
    save_format = PREPROCESSING_CONFIG["save_format"]

    # Save text features
    np.savez(
        os.path.join(save_dir, "text_features.npz"),
        features=text_features_all,
        labels=labels_all,
    )

    # Save image features
    np.savez(
        os.path.join(save_dir, "image_features.npz"),
        features=image_features_all,
        labels=labels_all,
    )

    # Save combined
    np.savez(
        os.path.join(save_dir, "combined_dataset.npz"),
        text_features=text_features_all,
        image_features=image_features_all,
        labels=labels_all,
    )

    # Save metadata
    metadata = {
        "num_samples": len(labels_all),
        "text_feature_shape": text_features_all.shape,
        "image_feature_shape": image_features_all.shape,
        "num_classes": len(set(labels_all)),
        "batch_size": PREPROCESSING_CONFIG["batch_size"],
    }

    with open(os.path.join(save_dir, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("✅ All data saved!")
    print(f"💾 Location: {save_dir}")

    # Cleanup
    del all_text_features, all_image_features
    gc.collect()
else:
    print("❌ No features to save")

✅ All data saved!
💾 Location: ./processed_data/crawled


In [11]:
# Final status
print("🚀 Preprocessing Complete!")
print("=" * 40)

save_dir = Path(PREPROCESSING_CONFIG["output_dir"])
if save_dir.exists():
    files = list(save_dir.glob("*"))
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"📁 {f.name}: {size_mb:.2f} MB")

print("\n🎯 Next step: Use processed data for model training")

🚀 Preprocessing Complete!
📁 text_features.npz: 7221.04 MB
📁 metadata.json: 0.00 MB
📁 image_features.npz: 37.65 MB
📁 combined_dataset.npz: 7258.65 MB

🎯 Next step: Use processed data for model training
